# LNR LNR stock — The Forecast Tool (Notebook 5)

Notebook 2 built a **progressive capability staircase** for the analyst agent:
no tools → news → news + open-ended code execution. This notebook adds a
**fourth, contrasting capability level**: a conventional **function tool**.

Instead of letting the agent write arbitrary Python, we expose a single,
rigidly-typed callable — `run_forecast` — that fits a pre-specified statistical
model (**AutoARIMA**) up to a cutoff date and returns a structured forecast. The
agent expresses intent through the tool's parameters (`series_id`,
`cutoff_date`, `horizons`, `frequency`); the series data never enters the LLM
context window.

| Path | Mechanism | Trade-off |
|------|-----------|-----------|
| `build_lnr_code_exec_config` | Open-ended code generation | Maximum flexibility, less control |
| `build_lnr_tool_config` (this NB) | Fixed function tool | Less flexibility, full control + reproducibility |

> **Prerequisite:** Read [`02_intro_agentic_predictor.ipynb`](02_intro_agentic_predictor.ipynb)
> first for the staircase framing and the `AgentPredictor` interface.

---
## 1. Setup & Data Registration

In [1]:
import sys
from pathlib import Path

ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "implementations").is_dir()),
    Path.cwd().resolve(),
)
sys.path[:0] = [str(ROOT / "aieng-forecasting"), str(ROOT / "implementations")]




import pandas as pd
from aieng.forecasting.evaluation.task import ForecastingTask
from energy_oil_for_lnr.data import LNR_SERIES_ID, build_lnr_service


# The data service is shared by the tool and the prompt builder. The tool reads
# series data directly from it (server-side) — it is never sent to the model.
data_service = build_lnr_service()

AS_OF = pd.Timestamp("2026-03-01")  # information cutoff (context available)
ORIGIN = pd.Timestamp("2026-03-02")  # the day we forecast *from*

ctx = data_service.context(as_of=AS_OF)
full_df = ctx.get_series(LNR_SERIES_ID)

print(f"Trading days in cache up to {AS_OF.date()}: {len(full_df)}")
print(f"Last LNR close: ${full_df['value'].iloc[-1]:.2f}/share on {str(full_df['timestamp'].iloc[-1])[:10]}")

Trading days in cache up to 2026-03-01: 2800
Last LNR close: $93.15/share on 2026-02-27


In [2]:
task = ForecastingTask(
    task_id="lnr_oil_price_forecast",
    target_series_id=LNR_SERIES_ID,
    horizons=[5, 10, 21],
    frequency="B",
    description="LNR LNR stock front-month futures — 5, 10, 21 business days ahead.",
)

print("Task:", task.task_id, "| horizons:", task.horizons, "| as_of:", ctx.as_of)

Task: lnr_oil_price_forecast | horizons: [5, 10, 21] | as_of: 2026-03-01 00:00:00


---
## 2. The tool, standalone

`ForecastTool` is deterministic and needs no LLM. We call it directly here to
show exactly what the agent will receive: a JSON block with point forecasts and
prediction intervals per horizon, plus the series metadata and the cutoff date
used.

We pass an explicit `data_service` (the one registered above) so the tool reads
from the same cache. The tool wraps a `Predictor`; here we inject an AutoARIMA
predictor with a modest `num_samples` (it is slow per origin).

In [3]:
from aieng.forecasting.methods.agentic import ForecastTool
from aieng.forecasting.methods.numerical.darts_arima import DartsAutoARIMAPredictor


tool = ForecastTool(data_service, predictor=DartsAutoARIMAPredictor(num_samples=200))

print("Running AutoARIMA forecast (this can take tens of seconds)...")
result_json = tool.run_forecast(
    series_id=LNR_SERIES_ID,
    cutoff_date=str(AS_OF.date()),
    horizons=task.horizons,
    frequency="B",
)
print(result_json)

Running AutoARIMA forecast (this can take tens of seconds)...
{
  "status": "ok",
  "series_id": "lnr_close_adj_cad",
  "series_description": "Linamar Corporation adjusted close (Yahoo Finance LNR.TO)",
  "units": "CAD/share",
  "frequency": "B",
  "cutoff_date": "2026-03-01",
  "n_observations_at_cutoff": 2800,
  "last_observed": {
    "date": "2026-02-27",
    "value": 93.14540100097656
  },
  "forecasts": [
    {
      "horizon": 5,
      "forecast_date": "2026-03-06",
      "point_forecast": 93.41586189568433,
      "intervals": {
        "80%": {
          "lower": 89.63516899297157,
          "upper": 96.85573541498657
        },
        "90%": {
          "lower": 89.0858558447285,
          "upper": 97.556714226892
        }
      },
      "quantiles": {
        "0.05": 89.0858558447285,
        "0.1": 89.63516899297157,
        "0.2": 90.93801502241077,
        "0.3": 91.63636429783826,
        "0.4": 92.26001402096308,
        "0.5": 93.41586189568433,
        "0.6": 94.14234

Note the `notes` field: a true 95% interval is not reported because the
predictor's standard quantile grid tops out at p05/p95, so the widest honest
interval is **90%** (p05–p95). The tool reports the **80%** (p10–p90) and
**90%** intervals plus the full quantile grid — it never fabricates coverage
the model did not produce.

---
## 3. Wiring the tool into the agent

`build_lnr_tool_config()` is the fourth capability factory. It combines the
bounded Google Search sub-agent (temporal cutoff enforced) with the forecast
tool, and appends an instruction supplement telling the agent to call
`run_forecast` once before producing its forecast.

We pass the same `data_service` so the config does not rebuild it.

In [4]:
from energy_oil_for_lnr.analyst_agent import (
    build_lnr_agent_predictor,
    build_lnr_tool_config,
)


# Models: "gemini-3.1-flash-lite-preview" (lite/default) · "gemini-3.5-flash" (advanced)
tool_config = build_lnr_tool_config(
    model="gemini-3.1-flash-lite-preview",
    # model="gemini-3.5-flash",  # advanced
    data_service=data_service,
    num_samples=200,
)

print("=== Tool config summary ===")
print("name:                ", tool_config.name)
print("model:               ", tool_config.model)
print("function_tools:      ", len(tool_config.function_tools))
print("context_retrieval:   ", tool_config.context_retrieval.enabled)
print("search_model:        ", tool_config.context_retrieval.search_model)
print("\n=== Forecast tool supplement (tail of instruction) ===")
print(tool_config.instruction[-700:])

=== Tool config summary ===
name:                 lnr_analyst_tool
model:                gemini-3.1-flash-lite-preview
function_tools:       1
context_retrieval:    True
search_model:         gemini-3.1-flash-lite-preview

=== Forecast tool supplement (tail of instruction) ===
 forecast you can reason from.

Call it ONCE before producing your forecast, with:
- `series_id`: "lnr_close_adj_cad"
- `cutoff_date`: the `as_of` date from the payload (YYYY-MM-DD). This is the
  information cutoff — the model uses only data on or before it.
- `horizons`: the `horizons` list from the payload.
- `frequency`: "B" (LNR trades on business days).

The tool returns JSON with point forecasts and 80%/90% prediction intervals per
horizon. Treat it as a disciplined statistical anchor: combine it with the
market context from the search sub-agent. You may adjust away from the baseline
when fundamentals or geopolitical risk justify it — document your reasoning in
the `rationale` fields.


---
## 4. A single agent call

Wrapping the config in an `AgentPredictor` and calling `predict` runs **one**
agent turn. In that turn the agent calls the Google Search sub-agent for market
context **and** `run_forecast` for the AutoARIMA anchor, then returns a
structured forecast that conditions on both.

In [5]:
tool_predictor = build_lnr_agent_predictor(tool_config)

print(f"Predictor ID: {tool_predictor.predictor_id}")
print("Running tool-equipped agent... (Google Search + AutoARIMA forecast tool)")

tool_preds = tool_predictor.predict(task, ctx)

print("\nTool-equipped agent forecast:\n")
for p in tool_preds:
    fc = p.payload
    print(
        f"  h={task.horizons[tool_preds.index(p)]:>2}d  "
        f"point=${fc.point_forecast:.2f}  "
        f"80%CI=[${fc.quantiles[0.10]:.2f}, ${fc.quantiles[0.90]:.2f}]"
    )
if tool_preds and tool_preds[0].metadata.get("rationale"):
    print("\nAgent rationale:", tool_preds[0].metadata["rationale"][:500])

Predictor ID: agent_predictor_lnr_analyst_tool_gemini-3.1-flash-lite-preview_continuous
Running tool-equipped agent... (Google Search + AutoARIMA forecast tool)


/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(



Tool-equipped agent forecast:

  h= 5d  point=$93.04  80%CI=[$89.39, $96.50]
  h=10d  point=$93.25  80%CI=[$89.19, $98.06]
  h=21d  point=$92.42  80%CI=[$85.29, $100.79]

Agent rationale: The forecast is grounded in the AutoARIMA statistical baseline, adjusted to reflect market sentiment surrounding the Persian Gulf geopolitical risk and OPEC+'s steady policy stance. As of March 1, 2026, the absence of active supply disruptions supports a stable to slightly consolidating price outlook, while the possibility of escalation justifies the broader tails in the probabilistic distribution.


---
## 5. Wrap-up

- The tool-equipped agent returns standard `Prediction` objects, so it drops
  straight into the Notebook 4 backtest harness via
  `build_lnr_agent_predictor(build_lnr_tool_config(...))`.
- **Conventional tools vs. code generation** is a deliberate design divergence:
  the tool path trades flexibility for a fixed, auditable, reproducible
  interface — arguably a safer way to grant an agent a new ability.
- AutoARIMA is just one example. `ForecastTool` wraps any `Predictor` (passed at
  construction), so swapping in Prophet, ETS, or an ensemble needs no signature
  change. And `series_id` makes the tool reusable for food CPI, the BoC rate,
  and other registered series.